In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv(r"D:\practice files\uber.csv")

In [ ]:
df.head()

| Source (Author / Institution / Date) | Key Findings & Insights (Precise) |
| :--- | :--- |
| **C. K. Suryadevara** <br> *Wilmington University* <br> (Dec 2019) | • Ride demand concentrates heavily around localized geographic clusters.<br>• Pickup volumes fluctuate dynamically with weather changes and local events. |
| **P. Christensen, G. Nino, & A. Osman** <br> *NBER* <br> (June 2023 / Rev. Sept 2025) | • **Demand Boost:** 50% fare discount quadrupled usage and expanded rider mobility by 49%.<br>• **Gender Gap:** Females are highly price-sensitive (elasticity -1.47) vs. males (-0.60) due to public transit safety concerns.<br>

# hypothesis:
- more distance=more fare.
- more passenger = more fare
- Rides during peak hours  have higher fares due to surge pricing, regardless of distance or passenger count.
- Surge pricing is like a temporary price increase when too many people want Uber rides at the same time

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# null value in  two columns

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['dropoff_longitude', 'dropoff_latitude'], how='any', inplace=True)

In [ ]:
df.isnull().sum()             # null values removed 

In [ ]:
df.drop(['Unnamed: 0', 'key'], axis=1, inplace=True)  

In [ ]:
df

removed two columns as they were not useful for analysis

In [ ]:
df.duplicated().sum()   # duplicates found 

# FEATURE ENGINEERING

In [ ]:

df['pickup_datetime'] = pd.to_datetime(
    df['pickup_datetime']
)

In [ ]:
df['pickup_datetime'].dtype

In [ ]:
# Extract useful time features:

df['year'] = df['pickup_datetime'].dt.year

df['month'] = df['pickup_datetime'].dt.month

df['day'] = df['pickup_datetime'].dt.day

df['hour'] = df['pickup_datetime'].dt.hour

df['weekday'] = df['pickup_datetime'].dt.day_name()


In [ ]:
from math import radians, sin, cos, sqrt, asin

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the distance (in km) between two points on Earth.
    """
   
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

   
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  
    return c * r


df['distance_km'] = df.apply(
    lambda row: haversine(
        row['pickup_longitude'],
        row['pickup_latitude'],
        row['dropoff_longitude'],
        row['dropoff_latitude']
    ),
    axis=1
)


print(df['distance_km'].head())

In [ ]:
df.columns

In [ ]:
for columns in df.columns:
    print(f"\n\033[1;30m{columns}\033[0m") 
    print("-" * 30)                   
    print(df[columns].value_counts())
    print() 
    

In [ ]:
(df['distance_km'] <= 0).sum()

In [ ]:
5632/200000*100

In [ ]:
df = df[df['distance_km'] > 0]

In [ ]:
(df['distance_km'] <= 0).sum()      #   removed

In [ ]:
df['distance_km'].describe()

In [ ]:
(df['distance_km'] >= 50).sum()

In [ ]:
df = df[df['distance_km'] < 50]

In [ ]:
497/194367*100

In [ ]:
df = df[df['distance_km'] > 1]

Out of 194367 rides, 497 had a distance greater than 50 km. I removed these because:

1️⃣ They represent 0.255% of all rides — so they're not typical.

2️⃣ Some may be data errors (like wrong pickup/dropoff locations).


By focusing on rides under 50 km, we get clearer insights about everyday Uber usage in the city.

In [ ]:
df['fare_amount'].describe()

In [ ]:
df = df[df['fare_amount'] > 0]

In [ ]:
df['fare_amount'].describe()

In [ ]:
(df['fare_amount'] < 2.5).sum()

In [ ]:
df = df[df['fare_amount'] > 2.5]

In [ ]:
df['fare_amount'].describe()

 **i removed fares under 2.5 because Uber's published minimum fare in most US cities is $2.50–$4.50 (base fare + booking fee), so charges below this are unlikely to be standard paid rides.**

In [ ]:
df['passenger_count'].describe()

In [ ]:
df=df[df['passenger_count'] != 208.000000]

In [ ]:
(df['passenger_count'] == 0).sum()

In [ ]:
df = df[df['passenger_count'] > 0]

# Univariate analysis:

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    df['fare_amount'],

    bins=50,
    kde=True,

    color='royalblue'
)

plt.title(
    'Fare Amount Distribution'
)

plt.xlabel('Fare Amount')

plt.show()

- Most Uber rides fall within lower fare ranges.
- Extremely expensive rides are relatively rare outliers.
- Uber’s ride ecosystem is dominated by short trips.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.graphics.mosaicplot import mosaic

In [ ]:
counts = df['passenger_count'].value_counts().sort_index()
counts.plot(kind='bar', title='Passenger Count Distribution')
plt.xlabel('Passenger Count')
plt.ylabel('Frequency')
plt.show()

- Most rides involve 1–2 passengers.
- Large-group rides occur significantly less frequently.
- Uber usage is primarily centered around individual mobility.

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    x=df['distance_km'],

    color='orange'
)

plt.title(
    'Ride Distance Distribution'
)

plt.show()

- Most rides cover short travel distances.
- Several extreme distance rides exist.

In [ ]:
hourly_rides = (
    df['hour']
    .value_counts()
    .sort_index()
    .reset_index()
)

hourly_rides.columns = [
    'hour',
    'count'
]

fig = px.line(
    hourly_rides,

    x='hour',
    y='count',

    markers=True,

    title='Ride Activity Across Hours'
)

fig.show()

- Ride demand fluctuates significantly throughout the day.
- Late-night activity remains comparatively lower.
- Late night from midnight to 5 AM is the quietest time because almost everyone is asleep in bed.
- The rest of the morning and evening stays steadily busy with people doing regular daily activities like getting  for work,lunch etc.
- The absolute highest peak of the day happens at 6:00 PM and 7:00 PM (hours 18 and 19) because that is the exact moment the evening rush hour hits its maximum as everyone heads home from work or out for dinner.

In [ ]:
weekday_order = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

plt.figure(figsize=(12,6))

sns.countplot(
    data=df,

    x='weekday',

    order=weekday_order,

    palette='viridis'
)

plt.title(
    'Ride Distribution by Weekday'
)

plt.show()

- Ride activity varies across weekdays.
- Working days generally maintain stronger transportation demand.
- Weekend behavior reflects leisure-oriented mobility patterns.

In [ ]:
import plotly.express as px
fig_bar = px.bar(
    monthly_rides,
    x='month',
    y='count',
    title='Monthly Ride Distribution (Bar Chart)',
    text='count', 
    color='count',  
)


fig_bar.update_traces(textposition='outside')
fig_bar.show()

- Ride activity changes dynamically across months.
- Seasonal travel behavior influences demand fluctuations.

In [ ]:
plt.figure(figsize=(10,6))

sns.kdeplot(
    df['fare_amount'],

    fill=True,

    color='crimson'
)

plt.title(
    'Fare Density Distribution'
)

plt.show()

In [ ]:
df['fare_amount'].describe()

- Dense fare concentration exists in lower pricing ranges.
- High-fare rides form a very small proportion of the dataset.
- so most of the customers prefer affordable rides.

In [ ]:
plt.figure(figsize=(10,6))

sns.kdeplot(
    df['distance_km'],

    fill=True,

    color='teal'
)

plt.title(
    'Ride Distance Density'
)

plt.show()

- Short-distance rides dominate the platform.
- Long-distance trips are rare.

In [ ]:
df['distance_km'].describe()

# BIVARIATE ANALYSIS

In [ ]:
fig = px.scatter(
    df.sample(162944),

    x='distance_km',
    y='fare_amount',


    title='Distance vs Fare Amount'
)

fig.show()

In [ ]:
df['distance_km'].describe()

- Fare amount increases strongly with travel distance.
- Extremely high fares at short distances may indicate anomalies or surge pricing.
- Distance acts as the primary pricing driver in Uber rides.

In [ ]:
plt.figure(figsize=(14,6))

sns.boxplot(
    data=df,

    x='hour',
    y='fare_amount',

    palette='coolwarm'
)

plt.title(
    'Fare Distribution Across Hours'
)

plt.show()

- Fare variability increases during peak travel hours.
- Night and rush-hour rides show higher pricing dispersion.
- Demand fluctuations strongly influence fare behavior.

In [ ]:
fig = px.violin(
    df,

    x='passenger_count',
    y='fare_amount',

    color='passenger_count',

    box=True,

    title='Passenger Count vs Fare Amount'
)

fig.show()

- Solo and duo riders take the longest trips: The highest fares (up to 220+) come from groups of just 1 or 2 passengers.
- Groups of 4 are the cheapest: Rides with exactly 4 passengers are the most predictable and rarely ever exceed 100.
- Passenger count doesn't change the base cost: No matter how many people get in, the vast majority of rides still cost under 20.  

In [ ]:
weekday_fare = (
    df.groupby('weekday')['fare_amount']
    .mean()
    .reset_index()
)

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = px.bar(
    weekday_fare,
    x='weekday',
    y='fare_amount',
    color='fare_amount',
    title='Average Fare by Weekday',
    category_orders={'weekday': weekday_order} 
)

fig.show()

- Fares are incredibly stable all week: The average fare barely changes from day to day, hovering tightly between 11.95 and 12.66 regardless of the weekday.
- Sunday is the most expensive day: Sunday sees the highest average fare (around 12.70), likely due to longer leisure trips.
- Saturday is the cheapest day: Surprisingly, Saturday has the lowest average fare (dipping near 11.95).


In [ ]:
df.groupby('weekday')['fare_amount'].mean()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df,
    x='passenger_count',
    y='distance_km',
    palette='viridis',
    inner='quartile',  
    cut=0,            
    scale='width'    
)
plt.title('Distance Distribution by Passenger Count (Violin Plot)')
plt.xlabel('Passenger Count')
plt.ylabel('Distance (km)')
plt.show()

- Short trips dominate: Across all group sizes, the vast majority of rides are under 5 km.

- Solo riders travel the furthest: Single passengers take the longest trips (up to 50 km), causing those extreme $220+ fares.

- Groups of 4 take the shortest rides.

- Groups of 5 and 6 travel slightly further (up to 35 km), likely due to planned group events.

In [ ]:
monthly_distance = (
    df.groupby('month')['distance_km']
    .mean()
    .reset_index()
)

fig = px.line(
    monthly_distance,

    x='month',
    y='distance_km',

    markers=True,

    title='Average Ride Distance by Month'
)

fig.show()

- Ride distance fluctuates across months.
- Summer sees the longest trips: Ride distances peak sharply in May and June , hitting nearly 3.95 km.
- Winter has the shortest trips: Average distance drops to its lowest point in January  at roughly 3.69 km, likely due to cold weather keeping people local.

# multivariate analysis:

In [ ]:
import plotly.express as px

fig = px.scatter(
    df.sample(100000),
    x='distance_km',
    y='fare_amount',
    size='passenger_count',  
    color='hour',
    hover_name='passenger_count',
    title='Ride Behavior: Distance vs. Fare (Bubble Size = Passengers)',
    size_max=20
)
fig.show()

- Distance drives the price: As expected, there is a clear upward trend—longer distances generally lead to higher fare amounts.
- The bubbles at the far right (35 to 50 km) are mostly small dots, confirming that ultra-long distance trips are almost exclusivelytaken by 1 or 2 passengers.
- The most expensive ride (200+ for 38 km) happened in the early morning (0–5 AM), when Uber charges extra for late-night rides.
- Short-distance anomalies: There is a tiny bubble on the far left that cost over 200 for a trip under 5 km. This is highly likely a data error, a massive tip, or an extreme flat-rate mistake.

In [ ]:

df['fare_category'] = pd.cut(
    df['fare_amount'],
    
    bins=[0,10,30,60,500],

    labels=[
        'Low',
        'Medium',
        'High',
        'Very High'
    ]
)

fig = px.parallel_categories(
    df.sample(5000),

    dimensions=[
        'passenger_count',
        'weekday',
        'fare_category'
    ],

    color='hour',

    color_continuous_scale=
    px.colors.sequential.Viridis
)

fig.show()

- Solo riders dominate across all days: Single passengers  make up the massive web of lines spreading out across every single weekday and mostly result in Low to Medium fares.
- Large groups prefer weekends: If you look at larger groups (sizes 3, 5, and 6 at the bottom left), their lines heavily flow towards Friday, Saturday, and Sunday, keeping their weekday presence minimal.
- "High" and "Very High" fares are rare: The lines thinning out at the bottom right prove that High and Very High fares happen rarely, and they are almost exclusively fed by 1 and 2-passenger trips stretching across the week.

In [ ]:
important_cols = [
    'fare_amount',
    'distance_km',
    'passenger_count',
    'hour',
    'month'
]

plt.figure(figsize=(8,6))

sns.heatmap(
    df[important_cols].corr(),

    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title(
    'Important Feature Correlation Heatmap'
)

plt.show()

- Distance is the ultimate price driver: fare_amount and distance_km have a near-perfect positive correlation (0.90). As distance increases, the fare almost always goes up.

- Passenger count has zero impact on fares: The correlation between passenger_count and fare_amount is practically non-existent (0.01). Adding more people to the car does not increase the price.

- Time of day and month don't change the base math: Both hour and month show near-zero correlations with fare amount and distance. Overall pricing structures stay heavily tied to distance year-round and clock-round.

In [ ]:
cluster_cols = [
    'fare_amount',
    'distance_km',
    'passenger_count',
    'hour',
    'month'
]

sns.clustermap(
    df[cluster_cols].corr(),
    annot=True,
    cmap='coolwarm',
    figsize=(5, 5)  
)

plt.show()

- Distance completely rules the fare: fare_amount and distance_km share a massive red square (0.9). Longer trips mean higher prices, period.

- Passenger count means nothing to the price: The connection between passengers and fare is basically zero (0.013). Extra people don't cost extra money.

- Time and season have zero impact: Both hour and month sit in solid blue boxes against everything else, meaning the time of day or the time of year doesn't change how fares or distances work.

# REPORT

- The exploratory analysis reveals that Uber ride pricing is primarily driven by travel distance, while temporal demand patterns and ride behavior create secondary pricing variations.

- Among all variables, distance emerges as the strongest factor influencing fare amount.

**The analysis further shows that:**

- peak travel hours increase fare variability,
- most rides occur within short  travel ranges,
- passenger count has relatively limited pricing influence,
- and ride demand changes dynamically across weekdays and months.

**Key Business Findings**
>Strongest Fare Drivers
- Travel distance
- Peak-hour demand
- 
# Final Conclusion

The dataset demonstrates that Uber’s ride ecosystem is fundamentally built around:

distance-based pricing combined with
dynamic demand behavior.

>The analysis shows that:

- most rides are short-distance urban trips,
- fare escalation occurs mainly for long-distance rides,
- and peak-hour demand creates pricing variability.